# 02 · Componente supervisado
**Objetivo:** predecir si un cliente suscribirá el depósito a plazo (`y`).

## 1. Selección y justificación de algoritmos
| Modelo | Por qué es apropiado | Supuestos | Ventajas | Limitaciones |
|---|---|---|---|---|
| Regresión logística | Línea base interpretable | Relación lineal en el log-odds | Coeficientes interpretables, rápida | No captura interacciones |
| Random Forest | Maneja no linealidades y variables mixtas | Ninguno fuerte | Robusto, importancia de variables | Más lento, menos interpretable |
| Gradient Boosting (HistGB) | Suele ser el mejor en datos tabulares | Ninguno fuerte | Alta capacidad predictiva | Riesgo de sobreajuste, más hiperparámetros |

_(Ampliar/ajustar esta tabla con los argumentos del equipo; ver `docs/02_modelos_supervisados.md`.)_

In [ ]:
import sys
sys.path.append("..")   # permite `from src import ...` al ejecutar desde notebooks/

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config
from src.data_loader import get_train_test, load_data

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)

X_train, X_test, y_train, y_test = get_train_test()
print("Train:", X_train.shape, "| Test:", X_test.shape)

from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from src import supervised as sup

## 2. Validación cruzada (solo con entrenamiento)

In [ ]:
models = sup.get_models()
cv_results = sup.cross_validate_models(models, X_train, y_train, cv=5)
display(cv_results.round(3))
cv_results.to_csv(config.RESULTS_DIR / "supervisado_cv.csv")

## 3. Entrenamiento final y evaluación en prueba

In [ ]:
fitted, rows, cms = {}, {}, {}
for name, model in models.items():
    pipe = sup.make_pipeline(model).fit(X_train, y_train)
    fitted[name] = pipe
    rows[name], cms[name] = sup.evaluate(pipe, X_test, y_test)          # umbral 0.5
test_results = pd.DataFrame(rows).T
display(test_results.round(3))
test_results.to_csv(config.RESULTS_DIR / "supervisado_test.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, cm) in zip(axes, cms.items()):
    ConfusionMatrixDisplay(cm, display_labels=["no", "yes"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "sup_matrices_confusion.png", dpi=150); plt.show()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.5))
for name, pipe in fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=a1, name=name)
    PrecisionRecallDisplay.from_estimator(pipe, X_test, y_test, ax=a2, name=name)
a1.set_title("Curva ROC"); a2.set_title("Curva Precisión-Recall")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "sup_curvas.png", dpi=150); plt.show()

**Interpretación (completar):** ¿qué modelo es mejor y según qué métrica? ¿Qué errores cuestan más para el
banco (llamar a quien no comprará vs. dejar de llamar a quien sí comprará)? ¿Hay sobreajuste (CV vs prueba)?: …

## 4. Selección de variables y reducción de dimensionalidad
Comparamos, con validación cruzada sobre entrenamiento, cuatro estrategias aplicadas **después** del
preprocesamiento y **dentro** del pipeline (se ajustan solo con cada fold de entrenamiento):
sin selección, información mutua (k = 20), L1 (regresión logística con penalización L1) y PCA (95 % de varianza).

In [ ]:
sel_models = {k: models[k] for k in ["Regresión logística", "Gradient Boosting"]}   # un modelo lineal y uno de árboles
sel_results = sup.compare_selectors(sel_models, sup.get_selectors(), X_train, y_train, cv=5)
display(sel_results.round(3))
sel_results.to_csv(config.RESULTS_DIR / "supervisado_seleccion_variables.csv")

In [ ]:
# ¿Cuántas variables conserva cada selector? (ajustado con todo el entrenamiento)
for s_name, selector in sup.get_selectors().items():
    if selector is None:
        continue
    p = sup.make_pipeline(models["Regresión logística"], selector).fit(X_train, y_train)
    n_out = p.named_steps["select"].transform(p.named_steps["prep"].transform(X_train[:5])).shape[1]
    print(f"{s_name}: {n_out} variables/componentes")

**Interpretación (completar):** ¿la selección mejora, empeora o no cambia las métricas? ¿Qué variables se
descartan y es coherente con el EDA (información mutua)? ¿Para qué modelo ayuda más la reducción?
Decisión final sobre usar o no selección: …

## 5. Optimización de hiperparámetros
`RandomizedSearchCV` (15 combinaciones, validación cruzada de 3 folds) optimizando **PR-AUC**, la métrica más
informativa con clases desbalanceadas. La búsqueda usa solo el conjunto de entrenamiento. _(Puede tardar varios minutos.)_

In [ ]:
tuned, tune_rows, times = {}, {}, {}
for name, model in models.items():
    search, secs = sup.tune_model(name, model, X_train, y_train, n_iter=15, cv=3)
    tuned[name], times[name] = search, secs
    tune_rows[name] = {"mejor_pr_auc_cv": search.best_score_, "segundos": secs, **search.best_params_}
    print(f"{name}: PR-AUC CV = {search.best_score_:.3f} ({secs:.0f} s)")
display(pd.DataFrame(tune_rows).T)
pd.DataFrame(tune_rows).T.to_csv(config.RESULTS_DIR / "supervisado_hiperparametros.csv")

In [ ]:
# Línea base vs optimizado, en prueba
rows_tuned = {}
for name, search in tuned.items():
    rows_tuned[name], _ = sup.evaluate(search.best_estimator_, X_test, y_test)
tuned_results = pd.DataFrame(rows_tuned).T
comparison = pd.concat({"base": test_results, "optimizado": tuned_results}, axis=1)
display(comparison.round(3))
comparison.to_csv(config.RESULTS_DIR / "supervisado_base_vs_optimizado.csv")

**Interpretación (completar):** ¿la optimización mejora la línea base? ¿Cuánto y para qué modelo? Si la mejora
es pequeña, ¿qué indica sobre el límite de la información disponible (sin `duration`)?: …

## 6. Comparación y selección del modelo
**Criterios (definidos antes de mirar los resultados):**
1. **PR-AUC y F1 en prueba** (principales, por el desbalance).
2. **Recall**: el banco prefiere no perder clientes interesados, siempre que la Precision no se desplome.
3. **Estabilidad**: diferencia entre PR-AUC de CV y de prueba (sobreajuste).
4. **Interpretabilidad** y **costo computacional** (tiempo de búsqueda/entrenamiento).

In [ ]:
ranking = pd.DataFrame({
    "pr_auc": tuned_results["pr_auc"], "f1": tuned_results["f1"], "recall": tuned_results["recall"],
    "roc_auc": tuned_results["roc_auc"],
    "brecha_cv_vs_test": [tuned[n].best_score_ - tuned_results.loc[n, "pr_auc"] for n in tuned_results.index],
    "segundos_busqueda": pd.Series(times),
})
ranking["rank_medio"] = ranking[["pr_auc", "f1"]].rank(ascending=False).mean(axis=1)
ranking = ranking.sort_values(["rank_medio", "pr_auc"], ascending=[True, False])   # empate -> mayor PR-AUC
display(ranking.round(3))
ranking.to_csv(config.RESULTS_DIR / "supervisado_ranking_modelos.csv")

best_name = ranking.index[0]          # ajustar manualmente si el equipo argumenta otro criterio
final_pipe = tuned[best_name].best_estimator_
print("Modelo seleccionado:", best_name)

**Selección argumentada (completar):** modelo elegido, por qué según cada criterio, y qué sacrificamos
(interpretabilidad, tiempo…). ¿Las diferencias entre modelos son significativas o están dentro del ruido de la CV?: …

## 7. Ajuste del umbral de decisión
Con clases desbalanceadas el umbral 0.5 rara vez es el mejor. Lo elegimos con **predicciones de
validación cruzada sobre entrenamiento** (nunca con prueba) y luego lo aplicamos a prueba.

In [ ]:
from sklearn.base import clone
skf = StratifiedKFold(5, shuffle=True, random_state=config.RANDOM_STATE)
oof = cross_val_predict(clone(final_pipe), X_train, y_train, cv=skf, method="predict_proba")[:, 1]
thr = sup.best_f1_threshold(y_train, oof)
print(f"Umbral óptimo (F1, validación): {thr:.3f}")
metrics_thr, cm_thr = sup.evaluate(final_pipe, X_test, y_test, threshold=thr)
display(pd.DataFrame({"umbral 0.5": tuned_results.loc[best_name], f"umbral {thr:.2f}": pd.Series(metrics_thr)}).round(3))
ConfusionMatrixDisplay(cm_thr, display_labels=["no", "yes"]).plot(cmap="Blues", colorbar=False)
plt.title(f"{best_name} · umbral {thr:.2f}")
plt.savefig(config.FIGURES_DIR / "sup_matriz_umbral_optimo.png", dpi=150, bbox_inches="tight"); plt.show()

**Interpretación (completar):** ¿qué cambia al mover el umbral (Recall ↑, Precision ↓)? Con costos supuestos
(p. ej. una llamada cuesta 1 y una suscripción vale 10), ¿qué umbral conviene al banco?: …

## 8. Importancia de variables (influencia de lo demográfico, financiero y de campaña)

In [ ]:
imp = sup.permutation_importances(final_pipe, X_test, y_test)
display(imp.round(4))
plt.figure(figsize=(7, 5))
sns.barplot(data=imp, y="variable", x="importancia", color="#4C72B0")
plt.title(f"Importancia por permutación (ROC-AUC) · {best_name}")
plt.savefig(config.FIGURES_DIR / "sup_importancia.png", dpi=150, bbox_inches="tight"); plt.show()
imp.to_csv(config.RESULTS_DIR / "supervisado_importancia.csv", index=False)

**Interpretación (completar):** variables más influyentes (¿demográficas, financieras o de campaña?),
coherencia con el EDA y limitaciones de interpretar importancia por permutación con variables correlacionadas: …

## 9. Conclusiones del componente supervisado (completar)
- ¿El modelo identifica correctamente a los clientes que aceptan? Métricas que lo respaldan: …
- Limitaciones de las métricas (PR-AUC vs ROC-AUC con desbalance, umbral, costos de error, un solo split de prueba): …
- Dificultades (desbalance, exclusión de `duration`, umbral): …
- Recomendación para una aplicación real: …